
4.6 Project - edwgol5635 - 02/22/2026


In [1]:

# Load the dataset 
import pandas as pd

file_path = 'C:\\Files\\Excel\\Project\\Happiness_Development_Index.xlsx'

# Load worksheet
df_raw = pd.read_excel(file_path, sheet_name='HDI')
print('Loaded shape:', df_raw.shape)
df_raw.head()


Loaded shape: (190, 7)


,HDI Rank,Country,Human Development Index (HDI),Life expectancy at birth,Expected years of schooling,Mean years of schooling,Gross national income (GNI) per capita
0,1,Switzerland,0.962,83.9872,16.500299,13.85966,66933.00454
1,2,Norway,0.961,83.2339,18.185200,13.00363,64660.10622
2,3,Iceland,0.959,82.6782,19.163059,13.76717,55782.04981
3,4,"Hong Kong, China (SAR)",0.952,85.4734,17.278170,12.22621,62606.84540
4,5,Australia,0.951,84.5265,21.054590,12.72682,49238.43335


In [2]:

# Prepare columns basic cleaning 
import numpy as np

cols_map = {
    'Human Development Index (HDI) ': 'HDI',
    'Human Development Index (HDI)': 'HDI',
    'Life expectancy at birth': 'life_expectancy',
    'Expected years of schooling': 'expected_schooling',
    'Mean years of schooling': 'mean_schooling',
    'Gross national income (GNI) per capita': 'gni_per_capita'
}

df = df_raw.rename(columns=cols_map)

needed = ['HDI','life_expectancy','expected_schooling','mean_schooling','gni_per_capita']
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Identify categorical object columns 
cat_cols = [c for c in df.columns if df[c].dtype == 'object' and c not in ['HDI']]
print('Categorical columns detected:', cat_cols)

# drop rows with missing required fields
df_subset = df[needed + cat_cols].copy()
rows_before = len(df_subset)
df_subset = df_subset.dropna(subset=needed)
print(f"Dropped {rows_before - len(df_subset)} rows with NA in required fields")

# Simple validity clamps to avoid negative inputs 
for c in ['life_expectancy','expected_schooling','mean_schooling','gni_per_capita']:
    df_subset[c] = pd.to_numeric(df_subset[c], errors='coerce').clip(lower=0)

print('Prepared shape:', df_subset.shape)
df_subset.head()


Categorical columns detected: []
Dropped 0 rows with NA in required fields
Prepared shape: (190, 5)


,HDI,life_expectancy,expected_schooling,mean_schooling,gni_per_capita
0,0.962,83.9872,16.500299,13.85966,66933.00454
1,0.961,83.2339,18.185200,13.00363,64660.10622
2,0.959,82.6782,19.163059,13.76717,55782.04981
3,0.952,85.4734,17.278170,12.22621,62606.84540
4,0.951,84.5265,21.054590,12.72682,49238.43335


In [3]:

# One-hot encode categorical columns (
import pandas as pd

df_encoded = pd.get_dummies(df_subset, columns=[c for c in df_subset.columns if df_subset[c].dtype=='object'], drop_first=True)
print('Encoded shape:', df_encoded.shape)
df_encoded.head()


Encoded shape: (190, 5)


,HDI,life_expectancy,expected_schooling,mean_schooling,gni_per_capita
0,0.962,83.9872,16.500299,13.85966,66933.00454
1,0.961,83.2339,18.185200,13.00363,64660.10622
2,0.959,82.6782,19.163059,13.76717,55782.04981
3,0.952,85.4734,17.278170,12.22621,62606.84540
4,0.951,84.5265,21.054590,12.72682,49238.43335


In [4]:

#Train/test split 70/30
from sklearn.model_selection import train_test_split

feature_cols = ['life_expectancy','expected_schooling','mean_schooling','gni_per_capita']
X = df_encoded[feature_cols]
y = df_encoded['HDI']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print('Train:', X_train.shape, y_train.shape)
print('Test:', X_test.shape, y_test.shape)


Train: (133, 4) (133,)
Test: (57, 4) (57,)


In [5]:

# Train Linear Regression 
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train, y_train)

print('Intercept:', linreg.intercept_)
print('Coefficients :', dict(zip(X_train.columns, linreg.coef_)))


Intercept: -0.050628452276883174
Coefficients : {'life_expectancy': np.float64(0.005868280500319557), 'expected_schooling': np.float64(0.012360570458971371), 'mean_schooling': np.float64(0.017782254610411174), 'gni_per_capita': np.float64(1.3254568188396376e-06)}


In [6]:

# Predictions
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

preds = linreg.predict(X_test)
mae = mean_absolute_error(y_test, preds)
mse = mean_squared_error(y_test, preds)
r2  = r2_score(y_test, preds)

print(f"MAE: {mae:.6f}")
print(f"MSE: {mse:.6f}")
print(f"R-squared: {r2:.6f}")


MAE: 0.016027
MSE: 0.000565
R-squared: 0.970315


In [7]:

# Feature importance 
import pandas as pd

coef_series = pd.Series(linreg.coef_, index=X_train.columns)
feat_importance = coef_series.abs().sort_values(ascending=False)
print('Feature importance:')
print(feat_importance)


Feature importance:
mean_schooling        0.017782
expected_schooling    0.012361
life_expectancy       0.005868
gni_per_capita        0.000001
dtype: float64
